In [1]:
import requests
import os
import sys
import platform
from lakehouse.spark import bronze, silver
from pyspark.sql import DataFrame, SparkSession
from delta import DeltaTable, configure_spark_with_delta_pip
from delta.tables import DeltaMergeBuilder
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
import json

In [2]:
if platform.system() == "Windows":
    os.environ["PYSPARK_PYTHON"] = sys.executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
    print("Adding Python ENV variables on Windows")

Adding Python ENV variables on Windows


In [3]:
builder = (
    SparkSession.builder.appName("Data with Nikk the Greek Spark Session")
    .master("local[4]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [4]:
CATALOG = spark.catalog.currentCatalog()

# 1. Set Up and Bronze Data

In [5]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")

DataFrame[]

In [6]:
options = {"catalog": CATALOG, "target_schema": "bronze"}

In [7]:
@F.udf(returnType="STRING")
def get_properties(url):
    json_request = requests.get(url).json()
    return json.dumps(json_request["result"]["properties"])

In [8]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return spark.createDataFrame(results)

    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        return df.withColumn("properties", get_properties(F.col("url")))


bronze_instance = StarWarsBronze(spark, **options)

In [9]:
bronze_instance.load().transform().write(mode="overwrite").execute("people", "planets")

2025-03-16 07:46:31 | people | execute | Started
2025-03-16 07:46:31 | people | execute | Started
2025-03-16 07:46:31 | people | load | Started
2025-03-16 07:46:37 | people | load | Completed in 0.08 min
2025-03-16 07:46:37 | people | transform | Started
2025-03-16 07:46:37 | people | transform | Completed in 0.0 min
2025-03-16 07:46:37 | people | write | Started
2025-03-16 07:47:00 | people | write | Completed in 0.38 min
2025-03-16 07:47:00 | people | execute | Completed in 0.47 min
2025-03-16 07:47:00 | planets | execute | Started
2025-03-16 07:47:00 | planets | load | Started
2025-03-16 07:47:03 | planets | load | Completed in 0.03 min
2025-03-16 07:47:03 | planets | transform | Started
2025-03-16 07:47:03 | planets | transform | Completed in 0.0 min
2025-03-16 07:47:03 | planets | write | Started
2025-03-16 07:47:18 | planets | write | Completed in 0.25 min
2025-03-16 07:47:18 | planets | execute | Completed in 0.3 min
2025-03-16 07:47:18 | people | execute | Completed in 0.77 min

In [10]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.people")
print(f"No. Rows: {df.count()}")
df.show()

No. Rows: 82
+--------------------+--------------------+---+--------------------+--------------------+
|         LH_BronzeTS|                name|uid|                 url|          properties|
+--------------------+--------------------+---+--------------------+--------------------+
|2025-03-16 07:46:...|      Luke Skywalker|  1|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:46:...|               C-3PO|  2|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:46:...|               R2-D2|  3|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:46:...|         Darth Vader|  4|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:46:...|         Leia Organa|  5|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:46:...|           Owen Lars|  6|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:46:...|  Beru Whitesun lars|  7|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:46:...|               R5-D4|  8|https://www.swapi...|{"created": "2025..

In [11]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.planets")
print(f"No. Rows: {df.count()}")
df.show(truncate=False)

No. Rows: 60
+--------------------------+--------------+---+-------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|LH_BronzeTS               |name          |uid|url                                  |properties                                                                                                                                                                                                                                                                                                                                                                                    |
+--------------------------+--------------+---+--

In [12]:
ddl_schema_planets = "diameter STRING, rotation_period STRING, orbital_period STRING, gravity STRING, population STRING, climate STRING, terrain STRING, surface_water STRING, created STRING, edited STRING, name STRING, url STRING"
ddl_schema_people = "height STRING, mass STRING, hair_color STRING, skin_color STRING, eye_color STRING, birth_year STRING, gender STRING, created STRING, edited STRING, name STRING, homeworld STRING, url STRING"
DDL_SCHEMAS = {"planets": ddl_schema_planets, "people": ddl_schema_people}

In [13]:
options = {
    "catalog": CATALOG,
    "source_schema": "bronze",
    "target_schema": "silver",
}

# 2 Overwrite

In [14]:
# with load filter and transformation
class StarWarsSilver(silver.Silver):
    def custom_filter(self, df: DataFrame, table: str) -> DataFrame:
        return df.where("uid <= '25'")

    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        df = df.withColumn(
            "properties", F.from_json(F.col("properties"), DDL_SCHEMAS[table])
        )
        df = df.withColumn("uid", F.col("uid").cast("int"))
        return df


silver_instance = StarWarsSilver(spark, **options)

In [15]:
silver_instance.load(filter="custom").transform().write(mode="overwrite").execute(
    "people", "planets"
)

2025-03-16 07:47:22 | people | execute | Started
2025-03-16 07:47:22 | people | execute | Started
2025-03-16 07:47:22 | people | load | Started
2025-03-16 07:47:22 | people | load | Completed in 0.0 min
2025-03-16 07:47:22 | people | transform | Started
2025-03-16 07:47:22 | people | transform | Completed in 0.0 min
2025-03-16 07:47:22 | people | write | Started
2025-03-16 07:47:25 | people | write | Completed in 0.03 min
2025-03-16 07:47:25 | people | execute | Completed in 0.03 min
2025-03-16 07:47:25 | planets | execute | Started
2025-03-16 07:47:25 | planets | load | Started
2025-03-16 07:47:25 | planets | load | Completed in 0.0 min
2025-03-16 07:47:25 | planets | transform | Started
2025-03-16 07:47:25 | planets | transform | Completed in 0.0 min
2025-03-16 07:47:25 | planets | write | Started
2025-03-16 07:47:26 | planets | write | Completed in 0.02 min
2025-03-16 07:47:26 | planets | execute | Completed in 0.02 min
2025-03-16 07:47:26 | people | execute | Completed in 0.07 min


In [16]:
df = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {df.count()}")
df.show(100, truncate=False)

No. Rows: 17
+--------------------------+--------------------------+---------------------+---+------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|LH_SilverTS               |LH_BronzeTS               |name                 |uid|url                                 |properties                                                                                                                                                                                                                |
+--------------------------+--------------------------+---------------------+---+------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [17]:
df = spark.sql(f"SELECT * FROM {CATALOG}.silver.planets")
print(f"No. Rows: {df.count()}")
df.show(100, truncate=False)

No. Rows: 18
+-------------------------+--------------------------+--------------+---+-------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|LH_SilverTS              |LH_BronzeTS               |name          |uid|url                                  |properties                                                                                                                                                                                                       |
+-------------------------+--------------------------+--------------+---+-------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|2025-03-16 07:47:25.

In [18]:
# without load filter
silver_instance.load().transform().write(mode="overwrite").execute("people", "planets")

2025-03-16 07:47:29 | people | execute | Started
2025-03-16 07:47:29 | people | execute | Started
2025-03-16 07:47:29 | people | load | Started
2025-03-16 07:47:29 | people | load | Completed in 0.0 min
2025-03-16 07:47:29 | people | transform | Started
2025-03-16 07:47:29 | people | transform | Completed in 0.0 min
2025-03-16 07:47:29 | people | write | Started
2025-03-16 07:47:31 | people | write | Completed in 0.02 min
2025-03-16 07:47:31 | people | execute | Completed in 0.02 min
2025-03-16 07:47:31 | planets | execute | Started
2025-03-16 07:47:31 | planets | load | Started
2025-03-16 07:47:31 | planets | load | Completed in 0.0 min
2025-03-16 07:47:31 | planets | transform | Started
2025-03-16 07:47:31 | planets | transform | Completed in 0.0 min
2025-03-16 07:47:31 | planets | write | Started
2025-03-16 07:47:32 | planets | write | Completed in 0.02 min
2025-03-16 07:47:32 | planets | execute | Completed in 0.02 min
2025-03-16 07:47:32 | people | execute | Completed in 0.05 min


In [19]:
df = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {df.count()}")
df.show()

No. Rows: 82
+--------------------+--------------------+-------------------+---+--------------------+--------------------+
|         LH_SilverTS|         LH_BronzeTS|               name|uid|                 url|          properties|
+--------------------+--------------------+-------------------+---+--------------------+--------------------+
|2025-03-16 07:47:...|2025-03-16 07:46:...|        Cliegg Lars| 62|https://www.swapi...|{183, unknown, br...|
|2025-03-16 07:47:...|2025-03-16 07:46:...|  Poggle the Lesser| 63|https://www.swapi...|{183, 80, none, g...|
|2025-03-16 07:47:...|2025-03-16 07:46:...|    Luminara Unduli| 64|https://www.swapi...|{170, 56.2, black...|
|2025-03-16 07:47:...|2025-03-16 07:46:...|      Barriss Offee| 65|https://www.swapi...|{166, 50, black, ...|
|2025-03-16 07:47:...|2025-03-16 07:46:...|              Dormé| 66|https://www.swapi...|{165, unknown, br...|
|2025-03-16 07:47:...|2025-03-16 07:46:...|              Dooku| 67|https://www.swapi...|{193, 80, white, ..

In [20]:
df = spark.sql(f"SELECT * FROM {CATALOG}.silver.planets")
print(f"No. Rows: {df.count()}")
df.show()

No. Rows: 60
+--------------------+--------------------+--------------+---+--------------------+--------------------+
|         LH_SilverTS|         LH_BronzeTS|          name|uid|                 url|          properties|
+--------------------+--------------------+--------------+---+--------------------+--------------------+
|2025-03-16 07:47:...|2025-03-16 07:47:...|       Mygeeto| 16|https://www.swapi...|{10088, 12, 167, ...|
|2025-03-16 07:47:...|2025-03-16 07:47:...|       Felucia| 17|https://www.swapi...|{9100, 34, 231, 0...|
|2025-03-16 07:47:...|2025-03-16 07:47:...|Cato Neimoidia| 18|https://www.swapi...|{0, 25, 278, 1 st...|
|2025-03-16 07:47:...|2025-03-16 07:47:...|     Saleucami| 19|https://www.swapi...|{14920, 26, 392, ...|
|2025-03-16 07:47:...|2025-03-16 07:47:...|       Stewjon| 20|https://www.swapi...|{0, unknown, unkn...|
|2025-03-16 07:47:...|2025-03-16 07:47:...|        Eriadu| 21|https://www.swapi...|{13490, 24, 360, ...|
|2025-03-16 07:47:...|2025-03-16 07:47:...

# 3 Replace Where

In [21]:
class StarWarsSilver(silver.Silver):
    def custom_filter(self, df: DataFrame, table: str) -> DataFrame:
        return df.where("uid > '0'")

    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        df = df.withColumn(
            "properties", F.from_json(F.col("properties"), DDL_SCHEMAS[table])
        )
        df = df.withColumn("uid", F.col("uid").cast("int"))
        return df

    def get_replace_condition(self, df: DataFrame, table: str) -> str:
        return "uid > 0"


silver_instance = StarWarsSilver(spark, **options)
silver_instance.load(filter="custom").transform().write(mode="replace").execute(
    "people", "planets"
)

2025-03-16 07:47:34 | people | execute | Started
2025-03-16 07:47:34 | people | execute | Started
2025-03-16 07:47:34 | people | load | Started
2025-03-16 07:47:34 | people | load | Completed in 0.0 min
2025-03-16 07:47:34 | people | transform | Started
2025-03-16 07:47:34 | people | transform | Completed in 0.0 min
2025-03-16 07:47:34 | people | write | Started
2025-03-16 07:47:37 | people | write | Completed in 0.03 min
2025-03-16 07:47:37 | people | execute | Completed in 0.03 min
2025-03-16 07:47:37 | planets | execute | Started
2025-03-16 07:47:37 | planets | load | Started
2025-03-16 07:47:37 | planets | load | Completed in 0.0 min
2025-03-16 07:47:37 | planets | transform | Started
2025-03-16 07:47:37 | planets | transform | Completed in 0.0 min
2025-03-16 07:47:37 | planets | write | Started
2025-03-16 07:47:39 | planets | write | Completed in 0.03 min
2025-03-16 07:47:39 | planets | execute | Completed in 0.03 min
2025-03-16 07:47:39 | people | execute | Completed in 0.07 min


In [22]:
df = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {df.count()}")
df.show()

No. Rows: 82
+--------------------+--------------------+--------------------+---+--------------------+--------------------+
|         LH_SilverTS|         LH_BronzeTS|                name|uid|                 url|          properties|
+--------------------+--------------------+--------------------+---+--------------------+--------------------+
|2025-03-16 07:47:...|2025-03-16 07:46:...|           Boba Fett| 22|https://www.swapi...|{183, 78.2, black...|
|2025-03-16 07:47:...|2025-03-16 07:46:...|               IG-88| 23|https://www.swapi...|{200, 140, none, ...|
|2025-03-16 07:47:...|2025-03-16 07:46:...|               Bossk| 24|https://www.swapi...|{190, 113, none, ...|
|2025-03-16 07:47:...|2025-03-16 07:46:...|    Lando Calrissian| 25|https://www.swapi...|{177, 79, black, ...|
|2025-03-16 07:47:...|2025-03-16 07:46:...|               Lobot| 26|https://www.swapi...|{175, 79, none, l...|
|2025-03-16 07:47:...|2025-03-16 07:46:...|              Ackbar| 27|https://www.swapi...|{180, 83, 

# 4 Append

In [23]:
class StarWarsSilver(silver.Silver):
    def custom_filter(self, df: DataFrame, table: str) -> DataFrame:
        return df.where("custom == 'custom'")

    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        df = df.withColumn(
            "properties", F.from_json(F.col("properties"), DDL_SCHEMAS[table])
        )
        df = df.withColumn("uid", F.col("uid").cast("int"))
        return df


silver_instance = StarWarsSilver(spark, **options)
silver_instance.load().transform().write().execute(
    "people", "planets"
)  # default mode is append

2025-03-16 07:47:40 | people | execute | Started
2025-03-16 07:47:40 | people | execute | Started
2025-03-16 07:47:40 | people | load | Started
2025-03-16 07:47:40 | people | load | Completed in 0.0 min
2025-03-16 07:47:40 | people | transform | Started
2025-03-16 07:47:40 | people | transform | Completed in 0.0 min
2025-03-16 07:47:40 | people | write | Started
2025-03-16 07:47:42 | people | write | Completed in 0.02 min
2025-03-16 07:47:42 | people | execute | Completed in 0.02 min
2025-03-16 07:47:42 | planets | execute | Started
2025-03-16 07:47:42 | planets | load | Started
2025-03-16 07:47:42 | planets | load | Completed in 0.0 min
2025-03-16 07:47:42 | planets | transform | Started
2025-03-16 07:47:42 | planets | transform | Completed in 0.0 min
2025-03-16 07:47:42 | planets | write | Started
2025-03-16 07:47:43 | planets | write | Completed in 0.02 min
2025-03-16 07:47:43 | planets | execute | Completed in 0.02 min
2025-03-16 07:47:43 | people | execute | Completed in 0.03 min


In [24]:
df = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {df.count()}")
df.show()

No. Rows: 164
+--------------------+--------------------+--------------------+---+--------------------+--------------------+
|         LH_SilverTS|         LH_BronzeTS|                name|uid|                 url|          properties|
+--------------------+--------------------+--------------------+---+--------------------+--------------------+
|2025-03-16 07:47:...|2025-03-16 07:46:...|           Boba Fett| 22|https://www.swapi...|{183, 78.2, black...|
|2025-03-16 07:47:...|2025-03-16 07:46:...|               IG-88| 23|https://www.swapi...|{200, 140, none, ...|
|2025-03-16 07:47:...|2025-03-16 07:46:...|               Bossk| 24|https://www.swapi...|{190, 113, none, ...|
|2025-03-16 07:47:...|2025-03-16 07:46:...|    Lando Calrissian| 25|https://www.swapi...|{177, 79, black, ...|
|2025-03-16 07:47:...|2025-03-16 07:46:...|               Lobot| 26|https://www.swapi...|{175, 79, none, l...|
|2025-03-16 07:47:...|2025-03-16 07:46:...|              Ackbar| 27|https://www.swapi...|{180, 83,

# 5 Merge

In [25]:
class StarWarsSilver(silver.Silver):
    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        df = df.withColumn(
            "properties", F.from_json(F.col("properties"), DDL_SCHEMAS[table])
        )
        df = df.withColumn("uid", F.col("uid").cast("int"))
        return df

    def get_delta_merge_builder(
        self, df: DataFrame, delta_table: DeltaTable
    ) -> DeltaMergeBuilder:
        merge_condition = "target.url = source.url"
        builder = delta_table.alias("target").merge(df.alias("source"), merge_condition)
        builder = builder.whenMatchedUpdateAll()
        builder = builder.whenNotMatchedInsertAll()
        return builder


silver_instance = StarWarsSilver(spark, **options)
silver_instance.load().transform().write(mode="merge").execute("people", "planets")

2025-03-16 07:47:44 | people | execute | Started
2025-03-16 07:47:44 | people | execute | Started
2025-03-16 07:47:44 | people | load | Started
2025-03-16 07:47:44 | people | load | Completed in 0.0 min
2025-03-16 07:47:44 | people | transform | Started
2025-03-16 07:47:44 | people | transform | Completed in 0.0 min
2025-03-16 07:47:44 | people | write | Started
2025-03-16 07:47:47 | people | write | Completed in 0.05 min
2025-03-16 07:47:47 | people | execute | Completed in 0.05 min
2025-03-16 07:47:47 | planets | execute | Started
2025-03-16 07:47:47 | planets | load | Started
2025-03-16 07:47:47 | planets | load | Completed in 0.0 min
2025-03-16 07:47:47 | planets | transform | Started
2025-03-16 07:47:47 | planets | transform | Completed in 0.0 min
2025-03-16 07:47:47 | planets | write | Started
2025-03-16 07:47:50 | planets | write | Completed in 0.03 min
2025-03-16 07:47:50 | planets | execute | Completed in 0.03 min
2025-03-16 07:47:50 | people | execute | Completed in 0.08 min


In [26]:
df = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {df.count()}")
df.show()

No. Rows: 164
+--------------------+--------------------+--------------------+---+--------------------+--------------------+
|         LH_SilverTS|         LH_BronzeTS|                name|uid|                 url|          properties|
+--------------------+--------------------+--------------------+---+--------------------+--------------------+
|2025-03-16 07:47:...|2025-03-16 07:46:...|      Luke Skywalker|  1|https://www.swapi...|{172, 77, blond, ...|
|2025-03-16 07:47:...|2025-03-16 07:46:...|      Luke Skywalker|  1|https://www.swapi...|{172, 77, blond, ...|
|2025-03-16 07:47:...|2025-03-16 07:46:...|      Obi-Wan Kenobi| 10|https://www.swapi...|{182, 77, auburn,...|
|2025-03-16 07:47:...|2025-03-16 07:46:...|      Obi-Wan Kenobi| 10|https://www.swapi...|{182, 77, auburn,...|
|2025-03-16 07:47:...|2025-03-16 07:46:...|    Anakin Skywalker| 11|https://www.swapi...|{188, 84, blond, ...|
|2025-03-16 07:47:...|2025-03-16 07:46:...|    Anakin Skywalker| 11|https://www.swapi...|{188, 84,

In [27]:
df = spark.sql(f"SELECT * FROM {CATALOG}.silver.planets")
print(f"No. Rows: {df.count()}")
df.show()

No. Rows: 120
+--------------------+--------------------+--------------+---+--------------------+--------------------+
|         LH_SilverTS|         LH_BronzeTS|          name|uid|                 url|          properties|
+--------------------+--------------------+--------------+---+--------------------+--------------------+
|2025-03-16 07:47:...|2025-03-16 07:47:...|      Tatooine|  1|https://www.swapi...|{10465, 23, 304, ...|
|2025-03-16 07:47:...|2025-03-16 07:47:...|      Tatooine|  1|https://www.swapi...|{10465, 23, 304, ...|
|2025-03-16 07:47:...|2025-03-16 07:47:...|        Kamino| 10|https://www.swapi...|{19720, 27, 463, ...|
|2025-03-16 07:47:...|2025-03-16 07:47:...|        Kamino| 10|https://www.swapi...|{19720, 27, 463, ...|
|2025-03-16 07:47:...|2025-03-16 07:47:...|      Geonosis| 11|https://www.swapi...|{11370, 30, 256, ...|
|2025-03-16 07:47:...|2025-03-16 07:47:...|      Geonosis| 11|https://www.swapi...|{11370, 30, 256, ...|
|2025-03-16 07:47:...|2025-03-16 07:47:..

# 6 Clean Up

In [ ]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.bronze CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.silver CASCADE")
spark.stop()

DataFrame[]